# Stacked Neural Network Ensemble for Landslide Size Classification
## Implementation with 10-Fold Cross-Validation

## 1. Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import random
import time
import warnings
warnings.filterwarnings('ignore')

from numpy import dstack, interp
from statistics import mean

# Keras
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Sklearn
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, auc
)

# XGBoost
from xgboost import XGBClassifier

# SMOTE
from imblearn.over_sampling import SMOTE

# ── Fix random seeds for reproducibility ──────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
print('All libraries imported and random seeds fixed.')

## 2. Load Dataset

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────
data = pd.read_csv('dataset.csv')
dataset = data.dropna(axis=0, how='any')

# Adjust column indices as needed:
# X → all feature columns (columns 3 to second-last)
# y → last column (landslide size label: 0=large, 1=medium, 2=small)
X = dataset.iloc[:, 3:-1].values
y = dataset.iloc[:, -1].values

print(f'Dataset shape: {X.shape}')
print(f'Class distribution before SMOTE: {dict(zip(*np.unique(y, return_counts=True)))}')

## 3. Helper Functions

In [ ]:
def build_ann1(input_dim):
    """ANN1: 2 hidden layers — relu activations."""
    model = Sequential([
        Dense(units=20, activation='relu', input_dim=input_dim),
        Dense(units=25, activation='relu'),
        Dense(units=3,  activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=['sparse_categorical_accuracy']   # metric for sparse labels
    )
    return model


def build_ann2(input_dim):
    """ANN2: 3 hidden layers — relu → relu → sigmoid."""
    model = Sequential([
        Dense(units=15, activation='relu',    input_dim=input_dim),
        Dense(units=20, activation='relu'),
        Dense(units=25, activation='sigmoid'),
        Dense(units=3,  activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=['sparse_categorical_accuracy']
    )
    return model


def build_ann3(input_dim):
    """ANN3: 3 hidden layers — relu → sigmoid → relu."""
    model = Sequential([
        Dense(units=20, activation='relu',    input_dim=input_dim),
        Dense(units=50, activation='sigmoid'),
        Dense(units=50, activation='relu'),
        Dense(units=3,  activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=['sparse_categorical_accuracy']
    )
    return model


def get_meta_learner(name):
    """Return the chosen meta-learner by name."""
    learners = {
        'KNN':    KNeighborsClassifier(n_neighbors=3, metric='minkowski', p=2),
        'SVM':    SVC(kernel='linear',  random_state=SEED, probability=True),
        'KSVM':   SVC(kernel='rbf',     random_state=SEED, probability=True),
        'NB':     GaussianNB(),
        'GB':     GradientBoostingClassifier(n_estimators=50, learning_rate=0.5,
                                              max_depth=6, random_state=SEED),
    }
    return learners[name]


def build_stacked_features(members, X):
    """Generate stacked feature matrix from base learner probability outputs."""
    stackX = None
    for model in members:
        yhat = model.predict(X, verbose=0)   # shape: (n_samples, 3)
        stackX = yhat if stackX is None else dstack((stackX, yhat))
    # Reshape from (n_samples, 3, n_models) → (n_samples, 3*n_models)
    stackX = stackX.reshape((stackX.shape[0], stackX.shape[1] * stackX.shape[2]))
    return stackX


def compute_roc_data(y_true, y_prob, n_classes=3):
    """Compute macro-averaged ROC curve data."""
    y_true_bin = pd.get_dummies(y_true).values
    fpr, tpr, roc_auc_dict = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc_dict[i] = auc(fpr[i], tpr[i])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)
    return all_fpr, mean_tpr, macro_auc


print('Helper functions defined.')

## 4. 10-Fold Cross-Validation with Within-Fold SMOTE

> SMOTE is applied *inside* each fold on the training split only.
> This prevents any synthetic samples from appearing in the test fold,
> eliminating the data leakage.

In [ ]:
# ── Choose meta-learner ───────────────────────────────────────────────────────
# Options: 'KNN', 'SVM', 'KSVM', 'NB', 'GB'
META_LEARNER_NAME = 'GB'

# ── Storage lists ─────────────────────────────────────────────────────────────
acc_train       = []
acc_test        = []
Precision_macro = []
Precision_micro = []
Recall_macro    = []
Recall_micro    = []
F1_macro        = []
F1_micro        = []
AUC_ovo         = []
AUC_ovr         = []
train_time      = []
test_time       = []
all_cms         = []   # confusion matrices per fold

# For final ROC curve (best fold by AUC)
best_y_test  = None
best_y_prob  = None
best_auc_val = -1

# ── 10-Fold Cross-Validation ──────────────────────────────────────────────────
kf = KFold(n_splits=10, shuffle=True, random_state=SEED)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f'\n--- Fold {fold_idx + 1} / 10 ---')

    # ── 4.1 Split into train and test ─────────────────────────────────────────
    X_train_raw, X_test_raw = X[train_index], X[test_index]
    y_train_raw, y_test_cv  = y[train_index], y[test_index]

    # ── 4.2 Apply SMOTE only on the training fold ─────────────────────────────
    # This is the corrected approach: SMOTE must not see test data
    smote = SMOTE(random_state=SEED)
    X_train_smote, y_train_cv = smote.fit_resample(X_train_raw, y_train_raw)
    print(f'  Class distribution after within-fold SMOTE: '
          f'{dict(zip(*np.unique(y_train_cv, return_counts=True)))}')

    # ── 4.3 Feature scaling ───────────────────────────────────────────────────
    # Scaler fitted on SMOTE-augmented training data, applied to test data
    sc = StandardScaler()
    X_train_cv = sc.fit_transform(X_train_smote)
    X_test_cv  = sc.transform(X_test_raw)

    input_dim = X_train_cv.shape[1]

    # ── 4.4 Train base learners (ANNs) ────────────────────────────────────────
    start = time.time()

    ann1 = build_ann1(input_dim)
    ann1.fit(X_train_cv, y_train_cv, batch_size=32, epochs=100, verbose=0)

    ann2 = build_ann2(input_dim)
    ann2.fit(X_train_cv, y_train_cv, batch_size=32, epochs=200, verbose=0)

    ann3 = build_ann3(input_dim)
    ann3.fit(X_train_cv, y_train_cv, batch_size=32, epochs=200, verbose=0)

    members = [ann1, ann2, ann3]

    # ── 4.5 Build stacked feature matrix and train meta-learner ───────────────
    # Use training data to generate stacked features for meta-learner fitting
    stacked_train = build_stacked_features(members, X_train_cv)
    meta_model = get_meta_learner(META_LEARNER_NAME)
    meta_model.fit(stacked_train, y_train_cv)

    end = time.time()
    t_train = (end - start) * 1000
    train_time.append(t_train)
    print(f'  Training time: {t_train:.2f} ms')

    # ── 4.6 Evaluate on test fold ─────────────────────────────────────────────
    start = time.time()

    stacked_test = build_stacked_features(members, X_test_cv)
    y_pred      = meta_model.predict(stacked_test)
    y_test_prob = meta_model.predict_proba(stacked_test)

    end = time.time()
    t_test = (end - start) * 1000
    test_time.append(t_test)
    print(f'  Testing time:  {t_test:.2f} ms')

    # Training predictions (for train accuracy)
    y_train_pred = meta_model.predict(stacked_train)

    # ── 4.7 Compute and store metrics ─────────────────────────────────────────
    cm = confusion_matrix(y_test_cv, y_pred)
    all_cms.append(cm)
    print(f'  Confusion Matrix:\n{cm}')

    acc_test.append(accuracy_score(y_test_cv,  y_pred))
    acc_train.append(accuracy_score(y_train_cv, y_train_pred))

    Precision_macro.append(precision_score(y_test_cv, y_pred, average='macro',  zero_division=0))
    Precision_micro.append(precision_score(y_test_cv, y_pred, average='micro',  zero_division=0))
    Recall_macro.append(recall_score(y_test_cv,    y_pred, average='macro',  zero_division=0))
    Recall_micro.append(recall_score(y_test_cv,    y_pred, average='micro',  zero_division=0))
    F1_macro.append(f1_score(y_test_cv,            y_pred, average='macro',  zero_division=0))
    F1_micro.append(f1_score(y_test_cv,            y_pred, average='micro',  zero_division=0))

    auc_ovo = roc_auc_score(y_test_cv, y_test_prob, multi_class='ovo')
    auc_ovr = roc_auc_score(y_test_cv, y_test_prob, multi_class='ovr')
    AUC_ovo.append(auc_ovo)
    AUC_ovr.append(auc_ovr)

    # Track best fold for ROC plot
    if auc_ovo > best_auc_val:
        best_auc_val = auc_ovo
        best_y_test  = y_test_cv
        best_y_prob  = y_test_prob

print('\n10-Fold Cross-Validation complete.')

## 5. Summary Results

In [ ]:
print('=' * 55)
print(f'  Meta-Learner : {META_LEARNER_NAME}')
print('=' * 55)
print(f'  Accuracy  (Train) : {mean(acc_train)*100:.2f} %')
print(f'  Accuracy  (Test)  : {mean(acc_test)*100:.2f} %')
print(f'  Std Dev   (Test)  : {np.std(acc_test)*100:.2f} %')
print(f'  Precision (macro) : {mean(Precision_macro):.3f}')
print(f'  Precision (micro) : {mean(Precision_micro):.3f}')
print(f'  Recall    (macro) : {mean(Recall_macro):.3f}')
print(f'  Recall    (micro) : {mean(Recall_micro):.3f}')
print(f'  F1-score  (macro) : {mean(F1_macro):.3f}')
print(f'  F1-score  (micro) : {mean(F1_micro):.3f}')
print(f'  AUC (OvO)         : {mean(AUC_ovo):.3f}')
print(f'  AUC (OvR)         : {mean(AUC_ovr):.3f}')
print(f'  Avg Train Time    : {mean(train_time):.2f} ms')
print(f'  Avg Test  Time    : {mean(test_time):.2f} ms')
print('=' * 55)

## 6. Save Per-Fold Results to CSV

In [ ]:
result_cv = pd.DataFrame({
    'Fold':              list(range(1, 11)),
    'Train_Accuracy':    [v * 100 for v in acc_train],
    'Test_Accuracy':     [v * 100 for v in acc_test],
    'Precision_macro':   Precision_macro,
    'Precision_micro':   Precision_micro,
    'Recall_macro':      Recall_macro,
    'Recall_micro':      Recall_micro,
    'F1_macro':          F1_macro,
    'F1_micro':          F1_micro,
    'AUC_ovo':           AUC_ovo,
    'AUC_ovr':           AUC_ovr,
    'Train_Time_ms':     train_time,
    'Test_Time_ms':      test_time
})

result_cv.to_csv(f'results_per_fold_{META_LEARNER_NAME}.csv', index=False)
print('Per-fold results saved.')
result_cv

## 7. Save Average Results to CSV

In [ ]:
result_avg = pd.DataFrame([{
    'Meta_Learner':       META_LEARNER_NAME,
    'Train_Accuracy':     mean(acc_train) * 100,
    'Test_Accuracy':      mean(acc_test)  * 100,
    'Std_Dev':            np.std(acc_test) * 100,
    'Precision_macro':    mean(Precision_macro),
    'Precision_micro':    mean(Precision_micro),
    'Recall_macro':       mean(Recall_macro),
    'Recall_micro':       mean(Recall_micro),
    'F1_macro':           mean(F1_macro),
    'F1_micro':           mean(F1_micro),
    'AUC_ovo':            mean(AUC_ovo),
    'AUC_ovr':            mean(AUC_ovr),
    'Avg_Train_Time_ms':  mean(train_time),
    'Avg_Test_Time_ms':   mean(test_time)
}])

result_avg.to_csv(f'results_average_{META_LEARNER_NAME}.csv', index=False)
print('Average results saved.')
result_avg

## 8. Aggregated Confusion Matrix (Sum Over All Folds)

In [ ]:
agg_cm = np.sum(all_cms, axis=0)
# Normalize to get proportions
agg_cm_norm = agg_cm.astype('float') / agg_cm.sum(axis=1)[:, np.newaxis]

print('Aggregated Confusion Matrix (counts):')
print(agg_cm)
print('\nNormalized Confusion Matrix (proportions):')
print(np.round(agg_cm_norm, 2))

# Save
pd.DataFrame(agg_cm,
             index=['Large (actual)', 'Medium (actual)', 'Small (actual)'],
             columns=['Large (pred)', 'Medium (pred)', 'Small (pred)']
).to_csv(f'confusion_matrix_{META_LEARNER_NAME}.csv')
print('Confusion matrix saved.')

## 9. ROC Curve Data (Best Fold by AUC)

In [ ]:
# Compute macro-averaged ROC using the best fold's predictions
all_fpr, mean_tpr, macro_auc = compute_roc_data(best_y_test, best_y_prob, n_classes=3)

print(f'Macro-averaged AUC (best fold): {macro_auc:.3f}')
print(f'Best fold AUC (OvO): {best_auc_val:.3f}')

# Save ROC data for plotting
roc_csv = pd.DataFrame({
    'FPR_Macro': all_fpr,
    'TPR_Macro': mean_tpr
})
roc_csv.to_csv(f'roc_data_{META_LEARNER_NAME}.csv', index=False)
print('ROC data saved.')